# Sesi 6 – Persiapan Data untuk Machine Learning

| Keterangan | Detail |
|---|---|
| **Nama Lengkap** | [Farraz Dirgham H] |
| **NIM** | [240401020170] |
| **Kelas** | [IF405] |
| **Mata Kuliah** | Data Science |
| **Topik** | Persiapan Data: Encoding, Scaling, Train-Test Split, Feature Engineering |

## Pendahuluan

Sesi 6 berfokus pada tahap **terakhir sebelum modeling**: mengubah data yang sudah bersih (Sesi 3) menjadi format yang siap dikonsumsi algoritma machine learning.

Pipeline persiapan data:
```
Data Bersih → Feature Engineering → Encoding → Scaling → Train-Test Split → Siap Model
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import (
    LabelEncoder, OneHotEncoder, OrdinalEncoder,
    StandardScaler, MinMaxScaler, RobustScaler
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

np.random.seed(42)
print('Library siap!')

## 1. Dataset: Data Karyawan Perusahaan

In [ ]:
n = 400
df = pd.DataFrame({
    'usia'          : np.random.randint(22, 58, n),
    'gaji_jt'       : np.random.normal(8, 2.5, n).clip(3).round(2),
    'pengalaman_th' : np.random.randint(0, 30, n),
    'pendidikan'    : np.random.choice(['SMA','D3','S1','S2','S3'], n,
                                        p=[0.10,0.15,0.50,0.20,0.05]),
    'departemen'    : np.random.choice(['IT','Finance','Marketing','HR','Operations'], n),
    'kota'          : np.random.choice(['Jakarta','Bandung','Surabaya','Medan'], n),
    'status_kerja'  : np.random.choice(['Tetap','Kontrak','Freelance'], n,
                                        p=[0.60,0.30,0.10]),
    'kepuasan'      : np.random.randint(1, 6, n),   # 1-5
    'lembur'        : np.random.choice([0, 1], n, p=[0.65, 0.35])
})

# Target: resign (1) atau tidak (0)
skor = (-df['kepuasan']*0.4 + df['lembur']*0.5
        - df['gaji_jt']*0.1 + np.random.normal(0, 0.5, n))
df['resign'] = (skor > skor.median()).astype(int)

# Tambah missing values
for col in ['gaji_jt', 'kepuasan']:
    df.loc[np.random.choice(df.index, 15, replace=False), col] = np.nan

print(f'Shape: {df.shape}')
print(f'Distribusi resign: {df["resign"].value_counts().to_dict()}')
df.head()

## 2. Feature Engineering – Membuat Fitur Baru dari yang Ada

In [ ]:
df_fe = df.copy()

# Rasio gaji per tahun pengalaman
df_fe['gaji_per_tahun_pengalaman'] = (df_fe['gaji_jt'] /
    (df_fe['pengalaman_th'] + 1)).round(3)

# Segmentasi usia
df_fe['segmen_usia'] = pd.cut(
    df_fe['usia'], bins=[21,30,40,50,60],
    labels=['Junior (22-30)','Mid (31-40)','Senior (41-50)','Expert (51+)']
)

# Fitur interaksi: lembur × kepuasan (burnout indicator)
df_fe['burnout_score'] = df_fe['lembur'] * (6 - df_fe['kepuasan'].fillna(3))

print('Fitur baru yang dibuat:')
print(df_fe[['gaji_jt','pengalaman_th','gaji_per_tahun_pengalaman',
             'usia','segmen_usia','lembur','kepuasan','burnout_score']].head(8))

## 3. Encoding Variabel Kategorikal

Tiga teknik encoding berbeda sesuai karakteristik data:

In [ ]:
df_enc = df_fe.copy()

# --- Teknik 1: Label Encoding (untuk variabel binary/ordinal) ---
le = LabelEncoder()
df_enc['status_encoded'] = le.fit_transform(df_enc['status_kerja'])
print('Label Encoding status_kerja:')
print(dict(zip(le.classes_, le.transform(le.classes_))))

# --- Teknik 2: Ordinal Encoding (untuk variabel dengan urutan jelas) ---
ordinal_enc = OrdinalEncoder(
    categories=[['SMA','D3','S1','S2','S3']]
)
df_enc['pendidikan_ord'] = ordinal_enc.fit_transform(df_enc[['pendidikan']])
print('\nOrdinal Encoding pendidikan:')
for orig, enc in zip(['SMA','D3','S1','S2','S3'], [0,1,2,3,4]):
    print(f'  {orig} → {enc}')

# --- Teknik 3: One-Hot Encoding (untuk variabel nominal tanpa urutan) ---
df_enc = pd.get_dummies(df_enc, columns=['departemen','kota'], drop_first=False, dtype=int)
new_cols = [c for c in df_enc.columns if 'departemen_' in c or 'kota_' in c]
print(f'\nOne-Hot Encoding → {len(new_cols)} kolom baru: {new_cols}')

## 4. Penanganan Missing Values dengan SimpleImputer

In [ ]:
print('Missing values sebelum imputasi:')
print(df_enc[['gaji_jt','kepuasan']].isnull().sum())

imputer_num = SimpleImputer(strategy='median')
df_enc[['gaji_jt','kepuasan']] = imputer_num.fit_transform(
    df_enc[['gaji_jt','kepuasan']]
)

print('\nMissing values setelah imputasi:')
print(df_enc[['gaji_jt','kepuasan']].isnull().sum())
print(f'Total missing: {df_enc.isnull().sum().sum()}')

## 5. Pemilihan Fitur (Feature Selection)

In [ ]:
# Pilih kolom numerik yang relevan untuk model
fitur_numerik = ['usia','gaji_jt','pengalaman_th','kepuasan','lembur',
                  'gaji_per_tahun_pengalaman','burnout_score','pendidikan_ord','status_encoded']
ohe_cols = [c for c in df_enc.columns if 'departemen_' in c or 'kota_' in c]

X = df_enc[fitur_numerik + ohe_cols]
y = df_enc['resign']

print(f'Shape fitur (X): {X.shape}')
print(f'Shape target (y): {y.shape}')
print(f'Kolom fitur: {list(X.columns)[:10]} ... ({len(X.columns)} total)')

## 6. Train-Test Split

In [ ]:
# 80% train, 20% test dengan stratifikasi target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('=== Train-Test Split (80:20) ===')
print(f'X_train : {X_train.shape}  |  y_train distribusi: {y_train.value_counts().to_dict()}')
print(f'X_test  : {X_test.shape}   |  y_test  distribusi: {y_test.value_counts().to_dict()}')

# Verifikasi proporsi terjaga
print(f'\nProporsi resign di train : {y_train.mean():.4f}')
print(f'Proporsi resign di test  : {y_test.mean():.4f}')

## 7. Scaling – Normalisasi & Standarisasi

**Penting:** Scaler di-fit HANYA pada data train, lalu diterapkan ke test.

In [ ]:
# Hanya kolom numerik utama yang perlu di-scale
cols_to_scale = ['usia','gaji_jt','pengalaman_th','kepuasan',
                  'gaji_per_tahun_pengalaman','burnout_score']

# --- Standard Scaler (Z-score) ---
std_scaler = StandardScaler()
X_train_std = X_train.copy()
X_test_std  = X_test.copy()
X_train_std[cols_to_scale] = std_scaler.fit_transform(X_train[cols_to_scale])
X_test_std[cols_to_scale]  = std_scaler.transform(X_test[cols_to_scale])  # hanya transform!

# --- MinMax Scaler ---
mm_scaler = MinMaxScaler()
X_train_mm = X_train.copy()
X_test_mm  = X_test.copy()
X_train_mm[cols_to_scale] = mm_scaler.fit_transform(X_train[cols_to_scale])
X_test_mm[cols_to_scale]  = mm_scaler.transform(X_test[cols_to_scale])

# --- Robust Scaler (tahan outlier) ---
rb_scaler = RobustScaler()
X_train_rb = X_train.copy()
X_test_rb  = X_test.copy()
X_train_rb[cols_to_scale] = rb_scaler.fit_transform(X_train[cols_to_scale])
X_test_rb[cols_to_scale]  = rb_scaler.transform(X_test[cols_to_scale])

print('Perbandingan nilai gaji_jt setelah scaling (5 sampel train):')
perbandingan = pd.DataFrame({
    'Original'  : X_train['gaji_jt'].head(5).values,
    'Standard'  : X_train_std['gaji_jt'].head(5).values.round(4),
    'MinMax'    : X_train_mm['gaji_jt'].head(5).values.round(4),
    'Robust'    : X_train_rb['gaji_jt'].head(5).values.round(4)
})
print(perbandingan)

## 8. Visualisasi Dampak Scaling

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

titles = ['Original', 'StandardScaler', 'MinMaxScaler', 'RobustScaler']
datasets = [X_train, X_train_std, X_train_mm, X_train_rb]
colors   = ['steelblue','#4CAF50','#FF5722','#9C27B0']

for ax, title, data, c in zip(axes, titles, datasets, colors):
    ax.hist(data['gaji_jt'], bins=25, color=c, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('gaji_jt')
    ax.set_ylabel('Frekuensi')
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Dampak Berbagai Teknik Scaling pada Distribusi Gaji',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('perbandingan_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. ColumnTransformer – Pipeline Preprocessing Terpadu

In [ ]:
# Mulai ulang dari data mentah untuk demonstrasi ColumnTransformer
X_raw = df[['usia','gaji_jt','pengalaman_th','kepuasan','lembur','pendidikan','departemen','kota']]
y_raw = df['resign']

X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y_raw, test_size=0.2,
                                            random_state=42, stratify=y_raw)

num_cols = ['usia','gaji_jt','pengalaman_th','kepuasan','lembur']
cat_ohe  = ['departemen','kota']
cat_ord  = ['pendidikan']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler' , StandardScaler())
])

cat_ohe_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

cat_ord_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(categories=[['SMA','D3','S1','S2','S3']]))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline    , num_cols),
    ('ohe', cat_ohe_pipeline, cat_ohe),
    ('ord', cat_ord_pipeline, cat_ord)
])

X_tr_prep = preprocessor.fit_transform(X_tr)
X_te_prep = preprocessor.transform(X_te)

print('=== ColumnTransformer Pipeline Terpadu ===')
print(f'Shape X_train setelah preprocessing: {X_tr_prep.shape}')
print(f'Shape X_test  setelah preprocessing: {X_te_prep.shape}')
print('\nData siap masuk ke model machine learning!')

## 10. Cross-Validation Split – Alternatif Evaluasi Robust

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('=== Stratified K-Fold (5 Fold) ===')
for fold, (train_idx, val_idx) in enumerate(skf.split(X_tr, y_tr), 1):
    train_fold = y_tr.iloc[train_idx]
    val_fold   = y_tr.iloc[val_idx]
    print(f'Fold {fold}: train={len(train_idx):3d} | val={len(val_idx):3d}'
          f' | resign_train={train_fold.mean():.3f} | resign_val={val_fold.mean():.3f}')

## 11. Kesimpulan

Pada Sesi 6 ini, saya menguasai seluruh pipeline persiapan data sebelum modeling:

| Tahap | Teknik | Alasan |
|---|---|---|
| Feature Engineering | Rasio, segmentasi, interaksi | Menambah informasi baru |
| Encoding Kategorik | Label, Ordinal, One-Hot | Mengubah teks menjadi angka |
| Imputasi | SimpleImputer (median) | Menangani sisa missing value |
| Train-Test Split | 80:20 + stratify | Evaluasi tidak bias |
| Scaling | Standard, MinMax, Robust | Menyamakan skala fitur |
| ColumnTransformer | Pipeline terpadu | Menghindari data leakage |
| Stratified K-Fold | 5 fold | Evaluasi lebih robust |

**Aturan emas yang wajib diingat:** Scaler dan Imputer harus di-**fit hanya pada data train**, lalu di-**transform pada data train & test**. Melanggar aturan ini menyebabkan *data leakage* yang membuat evaluasi model menjadi tidak valid.

> **Insight:** Persiapan data yang benar adalah penentu utama kualitas model. Model terbaik di dunia pun akan gagal jika diberi data yang tidak disiapkan dengan tepat.